In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
import os, re, gc, sys, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from dataclasses import dataclass
from collections import Counter
# =====================================================================
# 1. GLOBAL CONFIG
# =====================================================================
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_PROJECT"] = "smart-mcq-solver-challenge"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH  = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

MODEL_NAME  = "microsoft/deberta-v3-base"
MAX_LENGTH  = 224          # mild trim from 256 (was over-cut to 192 -> hurt convergence, reverted partially)
BATCH_SIZE  = 4            # raised from 2 now that fp16 frees up memory
GRAD_ACCUM  = 4            # effective batch size stays 16
EPOCHS_XFMR = 5            # RESTORED to 5 -- cutting to 3 clearly underfit on only ~1800 train rows
                           # (see log: 3-epoch val accuracy was only 0.405). fp16 + shorter MAX_LENGTH
                           # still keep this meaningfully faster than the very first version.
SBERT_NAME  = "sentence-transformers/all-MiniLM-L6-v2"  # tiny + CPU-friendly
option_to_index = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
index_to_option = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# =====================================================================
# 2. LOAD + CLEAN DATA
# =====================================================================
def clean_text(x):
    x = str(x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

train_df = pd.read_csv(TRAIN_PATH).fillna("")
test_df  = pd.read_csv(TEST_PATH).fillna("")

for col in ['prompt'] + OPTION_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col]  = test_df[col].apply(clean_text)

train_df['label'] = train_df['answer'].map(option_to_index)

# De-duplicate identical questions BEFORE splitting. If the same (or a near-identical)
# question appears twice, a stratified split can put one copy in train and the other in
# validation -- the model then "memorizes" it and validation looks artificially perfect
# without generalizing to the actual test set. Drop exact duplicates on the full question.
before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=['prompt'] + OPTION_COLS).reset_index(drop=True)
after_dedup = len(train_df)
print(f"Removed {before_dedup - after_dedup} duplicate question rows "
      f"({before_dedup} -> {after_dedup})")
# Stratified 90/10 split so validation class balance mirrors the full train set
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
tr_idx, val_idx = next(sss.split(train_df, train_df['label']))
df_train = train_df.iloc[tr_idx].reset_index(drop=True)
df_val   = train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train rows: {len(df_train)} | Val rows: {len(df_val)} | Test rows: {len(test_df)}")
print("Train class balance:\n", df_train['label'].value_counts(normalize=True).sort_index())

Removed 183 duplicate question rows (2000 -> 1817)
Train rows: 1635 | Val rows: 182 | Test rows: 500
Train class balance:
 label
0    0.180428
1    0.243425
2    0.233028
3    0.181040
4    0.162080
Name: proportion, dtype: float64


In [5]:
# =====================================================================
# 3. METRICS
# =====================================================================
def compute_map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for i, true in enumerate(labels):
        if true == top3[i, 0]: scores.append(1.0)
        elif true == top3[i, 1]: scores.append(0.5)
        elif true == top3[i, 2]: scores.append(1/3)
        else: scores.append(0.0)
    return float(np.mean(scores))

def compute_cls_metrics(logits, labels):
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average='macro')
    return acc, f1m

results = {}  # model_name -> dict(accuracy, macro_f1, map3)


In [6]:
# =====================================================================
# 4. MODEL 1 — FROM SCRATCH: Deep MLP over TF-IDF features
# =====================================================================
print("--- Training Model 1: Deep PyTorch MLP (From Scratch) ---")
wandb.init(project="smart-mcq-solver-challenge", name="Model_1_Scratch_DeepMLP", job_type="train", reinit=True)

def merge_text(df):
    return df['prompt'] + " [SEP] " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']

X_train_text = merge_text(df_train)
X_val_text   = merge_text(df_val)
X_test_text  = merge_text(test_df)

vectorizer = TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 3), sublinear_tf=True)
X_train_vec = torch.tensor(vectorizer.fit_transform(X_train_text).toarray(), dtype=torch.float32)
X_val_vec   = torch.tensor(vectorizer.transform(X_val_text).toarray(), dtype=torch.float32)
X_test_vec  = torch.tensor(vectorizer.transform(X_test_text).toarray(), dtype=torch.float32)

y_train_scratch = torch.tensor(df_train['label'].values, dtype=torch.long)
y_val_scratch   = torch.tensor(df_val['label'].values, dtype=torch.long)

scratch_loader = DataLoader(TensorDataset(X_train_vec, y_train_scratch), batch_size=64, shuffle=True)

class DeepMCQClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.LayerNorm(128), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(128, 5)
        )
    def forward(self, x):
        return self.network(x)

scratch_model = DeepMCQClassifier(X_train_vec.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
# Lower LR + higher weight decay than before: the previous settings (lr=2e-3, wd=0.01)
# let this model hit a suspicious 100% validation score, which then fooled the ensemble
# weight search into over-trusting it. This model has ~4000 input features but only
# ~1800 training rows, so it's very easy for it to overfit.
optimizer = torch.optim.AdamW(scratch_model.parameters(), lr=1e-3, weight_decay=0.05)

EPOCHS_SCRATCH = 15
best_val_map, best_epoch, best_state = -1.0, -1, None

for epoch in range(EPOCHS_SCRATCH):
    scratch_model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in scratch_loader:
        optimizer.zero_grad()
        loss = criterion(scratch_model(batch_X.to(device)), batch_y.to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scratch_model.eval()
    with torch.no_grad():
        val_logits_epoch = scratch_model(X_val_vec.to(device)).cpu().numpy()
    val_map = compute_map_at_3(val_logits_epoch, y_val_scratch.numpy())
    val_acc, val_f1 = compute_cls_metrics(val_logits_epoch, y_val_scratch.numpy())
    wandb.log({"epoch": epoch + 1, "scratch_loss": epoch_loss / len(scratch_loader),
               "scratch_val_map": val_map, "scratch_val_accuracy": val_acc, "scratch_val_f1": val_f1})
    # Checkpoint the best-performing epoch instead of always keeping the last one --
    # this stops a later, more-overfit epoch from silently becoming the model we ship.
    if val_map > best_val_map:
        best_val_map = val_map
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in scratch_model.state_dict().items()}

if best_state is not None:
    scratch_model.load_state_dict(best_state)
print(f"Restored Model 1 weights from epoch {best_epoch} (best val MAP@3: {best_val_map:.4f})")

with torch.no_grad():
    scratch_train_logits = scratch_model(X_train_vec.to(device)).cpu().numpy()
    scratch_val_logits   = scratch_model(X_val_vec.to(device)).cpu().numpy()
    scratch_val_preds    = torch.softmax(torch.tensor(scratch_val_logits), dim=1).numpy()
    scratch_test_preds   = torch.softmax(scratch_model(X_test_vec.to(device)), dim=1).detach().cpu().numpy()
# Diagnostic: compare train vs. val performance. A large gap (train >> val) is the
# signature of overfitting -- worth checking before trusting this model in the ensemble.
m1_train_acc, m1_train_f1 = compute_cls_metrics(scratch_train_logits, y_train_scratch.numpy())
m1_train_map = compute_map_at_3(scratch_train_logits, y_train_scratch.numpy())
m1_acc, m1_f1 = compute_cls_metrics(scratch_val_logits, y_val_scratch.numpy())
m1_map = compute_map_at_3(scratch_val_logits, y_val_scratch.numpy())
print(f"Model 1 [train] -> Acc: {m1_train_acc:.4f} | Macro-F1: {m1_train_f1:.4f} | MAP@3: {m1_train_map:.4f}")
print(f"Model 1 [val]   -> Acc: {m1_acc:.4f} | Macro-F1: {m1_f1:.4f} | MAP@3: {m1_map:.4f}")
if m1_train_map - m1_map > 0.15:
    print("WARNING: large train/val gap -- this model may still be overfitting.")

results['Model 1 - Scratch MLP'] = {'accuracy': m1_acc, 'macro_f1': m1_f1, 'map3': m1_map}
wandb.log({"final_accuracy": m1_acc, "final_macro_f1": m1_f1, "final_map3": m1_map,
           "train_accuracy": m1_train_acc, "train_map3": m1_train_map, "best_epoch": best_epoch})
wandb.finish()

# Free the (small) GPU footprint before the next model
del scratch_model
gc.collect(); torch.cuda.empty_cache()


wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.


--- Training Model 1: Deep PyTorch MLP (From Scratch) ---


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260802_005212-1qlvyql2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model_1_Scratch_DeepMLP
wandb: ⭐️ View project at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: 🚀 View run at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/1qlvyql2
wandb: updating run metadata


Restored Model 1 weights from epoch 1 (best val MAP@3: 1.0000)
Model 1 [train] -> Acc: 0.9994 | Macro-F1: 0.9994 | MAP@3: 0.9997
Model 1 [val]   -> Acc: 1.0000 | Macro-F1: 1.0000 | MAP@3: 1.0000


wandb: 
wandb: Run history:
wandb:           best_epoch ▁
wandb:                epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb:       final_accuracy ▁
wandb:       final_macro_f1 ▁
wandb:           final_map3 ▁
wandb:         scratch_loss █▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: scratch_val_accuracy ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       scratch_val_f1 ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      scratch_val_map ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:       train_accuracy ▁
wandb:                   +1 ...
wandb: 
wandb: Run summary:
wandb:           best_epoch 1
wandb:                epoch 15
wandb:       final_accuracy 1
wandb:       final_macro_f1 1
wandb:           final_map3 1
wandb:         scratch_loss 0.00108
wandb: scratch_val_accuracy 1
wandb:       scratch_val_f1 1
wandb:      scratch_val_map 1
wandb:       train_accuracy 0.99939
wandb:                   +1 ...
wandb: 
wandb: 🚀 View run Model_1_Scratch_DeepMLP at: https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/1qlvyql2
wandb: ⭐️ View project at: https://wandb.ai/23f3000333-dl

In [7]:
# =====================================================================
# 5. MODEL 2 — PRETRAINED: Fine-tuned DeBERTa-v3-base
# =====================================================================
print("\n--- Training Model 2: Fine-Tuned Pretrained DeBERTa ---")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def preprocess_function(examples):
    n = len(examples['prompt'])
    first_sentences, second_sentences = [], []
    for i in range(n):
        prompt_txt = f"Context Question: {examples['prompt'][i]}"
        for opt in OPTION_COLS:
            first_sentences.append(prompt_txt)
            second_sentences.append(f"Hypothesis Option: {examples[opt][i]}")
    tokenized = tokenizer(first_sentences, second_sentences, truncation=True,
                          max_length=MAX_LENGTH, padding='max_length')
    return {k: [v[i*5:(i+1)*5] for i in range(n)] for k, v in tokenized.items()}

def make_ds(df):
    drop_cols = [c for c in df.columns if c not in ['input_ids', 'attention_mask', 'token_type_ids', 'label']]
    return Dataset.from_pandas(df).map(preprocess_function, batched=True, batch_size=100, remove_columns=drop_cols)
tok_train = make_ds(df_train)
tok_val   = make_ds(df_val)
tok_test  = make_ds(test_df)

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    def __call__(self, features):
        has_labels = 'label' in features[0]
        labels = [int(f.pop('label')) for f in features] if has_labels else None
        bs, nc = len(features), len(features[0]['input_ids'])
        flat_features = [{k: v[i] for k, v in f.items()} for f in features for i in range(nc)]
        batch = self.tokenizer.pad(flat_features, padding=True, return_tensors='pt')
        batch = {k: v.view(bs, nc, -1) for k, v in batch.items()}
        if labels is not None:
            batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    acc, f1m = compute_cls_metrics(logits, labels)
    return {'map_at_3': compute_map_at_3(logits, labels), 'accuracy': acc, 'macro_f1': f1m}

label_counts = Counter(df_train['label'].values)
total_samples = len(df_train)
class_weights = torch.tensor([total_samples / (5 * label_counts.get(i, 1)) for i in range(5)],
                              dtype=torch.float).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if "labels" in inputs:
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss = torch.nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss
        outputs = model(**inputs)
        loss = outputs.loss if getattr(outputs, "loss", None) is not None else torch.tensor(0.0).to(device)
        return (loss, outputs) if return_outputs else loss

pt_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).float()

args = TrainingArguments(
    output_dir='./results',
    run_name="Model_2_Pretrained_DeBERTa",
    eval_strategy='epoch',
    save_strategy='no',           # avoids Kaggle disk exhaustion
    learning_rate=1e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS_XFMR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),   # ~2x speedup on Kaggle T4, negligible score impact
    report_to="wandb",
    gradient_checkpointing=True,
    seed=SEED,
    logging_steps=25,
)

trainer = WeightedTrainer(
    model=pt_model, args=args,
    train_dataset=tok_train, eval_dataset=tok_val,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer),
    compute_metrics=hf_compute_metrics
)

gc.collect(); torch.cuda.empty_cache()
trainer.train()

val_res  = trainer.predict(tok_val).predictions
test_res = trainer.predict(tok_test).predictions
val_logits  = val_res[0]  if isinstance(val_res, tuple)  else val_res
test_logits = test_res[0] if isinstance(test_res, tuple) else test_res

deberta_val_preds  = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
deberta_test_preds = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

m2_acc, m2_f1 = compute_cls_metrics(val_logits, df_val['label'].values)
m2_map = compute_map_at_3(val_logits, df_val['label'].values)
results['Model 2 - Pretrained DeBERTa'] = {'accuracy': m2_acc, 'macro_f1': m2_f1, 'map3': m2_map}
wandb.finish()
print(f"Model 2 -> Acc: {m2_acc:.4f} | Macro-F1: {m2_f1:.4f} | MAP@3: {m2_map:.4f}")

# IMPORTANT: DeBERTa + the Trainer's internal state stay resident on the GPU otherwise.
# Model 3 loads a second (smaller) model next -- free VRAM first to avoid a CUDA OOM.
del trainer, pt_model
gc.collect(); torch.cuda.empty_cache()


--- Training Model 2: Fine-Tuned Pretrained DeBERTa ---


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1635 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260802_005229-9b6yleb2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model_2_Pretrained_DeBERTa
wandb: ⭐️ View project at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: 🚀 View run at https://wandb.ai/23f3000333-dl-genai-project/smart-mc

Epoch,Training Loss,Validation Loss,Map At 3,Accuracy,Macro F1
1,6.347057,1.500711,0.583333,0.417582,0.408285
2,5.529996,1.237414,0.695971,0.549451,0.547490
3,4.670333,0.962838,0.781136,0.659341,0.660637
4,4.264525,0.819250,0.801282,0.681319,0.680554
5,4.067671,0.806819,0.808608,0.697802,0.694999


wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▄▇██
wandb:               eval/loss █▅▃▁▁
wandb:           eval/macro_f1 ▁▄▇██
wandb:           eval/map_at_3 ▁▅▇██
wandb:            eval/runtime ▁▆█▂▄
wandb: eval/samples_per_second █▃▁▇▅
wandb:   eval/steps_per_second █▃▁▇▅
wandb:           test/accuracy ▁
wandb:               test/loss ▁
wandb:           test/macro_f1 ▁
wandb:                      +9 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.6978
wandb:               eval/loss 0.80682
wandb:           eval/macro_f1 0.695
wandb:           eval/map_at_3 0.80861
wandb:            eval/runtime 9.245
wandb: eval/samples_per_second 19.686
wandb:   eval/steps_per_second 1.298
wandb:           test/accuracy 0.6978
wandb:               test/loss 0.80682
wandb:           test/macro_f1 0.695
wandb:                     +14 ...
wandb: 
wandb: 🚀 View run Model_2_Pretrained_DeBERTa at: https://wandb.ai/23f3000333-dl-genai-project/smart-

Model 2 -> Acc: 0.6978 | Macro-F1: 0.6950 | MAP@3: 0.8086


In [8]:
# =====================================================================
# 6. MODEL 3 — ADDITIONAL MODEL OF CHOICE: Sentence-Embedding + Logistic Regression
# =====================================================================
print("\n--- Training Model 3: Sentence-Embedding Similarity + Logistic Regression ---")
wandb.init(project="smart-mcq-solver-challenge", name="Model_3_Choice_SimilarityLR", job_type="train", reinit=True)

# Defensive import: not every Kaggle image ships sentence-transformers by default.
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer(SBERT_NAME, device=device)

def row_cosine_sim(a, b):
    """Row-wise cosine similarity between two equal-shaped dense arrays (O(n), not O(n^2))."""
    dot = np.sum(a * b, axis=1)
    denom = (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)) + 1e-8
    return dot / denom

def row_cosine_sim_sparse(a_sparse, b_sparse):
    """Row-wise cosine similarity between two equal-shaped sparse matrices (O(n), not O(n^2))."""
    dot = np.asarray(a_sparse.multiply(b_sparse).sum(axis=1)).flatten()
    a_norm = np.sqrt(np.asarray(a_sparse.multiply(a_sparse).sum(axis=1)).flatten())
    b_norm = np.sqrt(np.asarray(b_sparse.multiply(b_sparse).sum(axis=1)).flatten())
    return dot / (a_norm * b_norm + 1e-8)

def build_similarity_features(df, vectorizer, sbert_model):
    prompt_emb = sbert_model.encode(df['prompt'].tolist(), batch_size=64, show_progress_bar=False)
    prompt_tfidf = vectorizer.transform(df['prompt'])

    sbert_feats, tfidf_feats = [], []
    for opt in OPTION_COLS:
        opt_emb = sbert_model.encode(df[opt].tolist(), batch_size=64, show_progress_bar=False)
        sbert_feats.append(row_cosine_sim(prompt_emb, opt_emb))

        opt_tfidf = vectorizer.transform(df[opt])
        tfidf_feats.append(row_cosine_sim_sparse(prompt_tfidf, opt_tfidf))

    return np.column_stack(sbert_feats + tfidf_feats)  # shape: (n, 10)

X_train_sim = build_similarity_features(df_train, vectorizer, sbert)
X_val_sim   = build_similarity_features(df_val, vectorizer, sbert)
X_test_sim  = build_similarity_features(test_df, vectorizer, sbert)

logreg = LogisticRegression(max_iter=2000, C=2.0, random_state=SEED)
logreg.fit(X_train_sim, df_train['label'].values)

logreg_val_preds  = logreg.predict_proba(X_val_sim)
logreg_test_preds = logreg.predict_proba(X_test_sim)

m3_acc, m3_f1 = compute_cls_metrics(logreg_val_preds, df_val['label'].values)
m3_map = compute_map_at_3(logreg_val_preds, df_val['label'].values)
results['Model 3 - Similarity + LogReg'] = {'accuracy': m3_acc, 'macro_f1': m3_f1, 'map3': m3_map}
wandb.log({"final_accuracy": m3_acc, "final_macro_f1": m3_f1, "final_map3": m3_map})
wandb.finish()
print(f"Model 3 -> Acc: {m3_acc:.4f} | Macro-F1: {m3_f1:.4f} | MAP@3: {m3_map:.4f}")

del sbert
gc.collect(); torch.cuda.empty_cache()

wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.



--- Training Model 3: Sentence-Embedding Similarity + Logistic Regression ---


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260802_012023-rw3pap18
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model_3_Choice_SimilarityLR
wandb: ⭐️ View project at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: 🚀 View run at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/rw3pap18


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: final_accuracy ▁
wandb: final_macro_f1 ▁
wandb:     final_map3 ▁
wandb: 
wandb: Run summary:
wandb: final_accuracy 0.28022
wandb: final_macro_f1 0.26442
wandb:     final_map3 0.49267
wandb: 
wandb: 🚀 View run Model_3_Choice_SimilarityLR at: https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/rw3pap18
wandb: ⭐️ View project at: https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260802_012023-rw3pap18/logs


Model 3 -> Acc: 0.2802 | Macro-F1: 0.2644 | MAP@3: 0.4927


In [9]:
# =====================================================================
# 7. MODEL COMPARISON (3 core models, per the guideline)
# =====================================================================
comparison_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
comparison_df = comparison_df[['model', 'accuracy', 'macro_f1', 'map3']]
print(comparison_df.to_string(index=False))

wandb.init(project="smart-mcq-solver-challenge", name="Model_Comparison_Summary", job_type="analysis", reinit=True)
wandb.log({"comparison_table": wandb.Table(dataframe=comparison_df)})
for _, row in comparison_df.iterrows():
    wandb.summary[f"{row['model']}_accuracy"] = row['accuracy']
    wandb.summary[f"{row['model']}_macro_f1"] = row['macro_f1']
    wandb.summary[f"{row['model']}_map3"] = row['map3']
wandb.finish()

wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.


                        model  accuracy  macro_f1     map3
        Model 1 - Scratch MLP  1.000000  1.000000 1.000000
 Model 2 - Pretrained DeBERTa  0.697802  0.694999 0.808608
Model 3 - Similarity + LogReg  0.280220  0.264424 0.492674


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260802_012035-gueqnnfb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model_Comparison_Summary
wandb: ⭐️ View project at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: 🚀 View run at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/gueqnnfb
wandb: updating run metadata; uploading artifact run-gueqnnfb-comparison_table; uploading summary
wandb: uploading artifact run-gueqnnfb-comparison_table
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run summary:
wandb:         Model 1 - Scratch MLP_accuracy 1
wandb:         Model 1 - Scratch MLP_macro_f1 1
wandb:             Model 1 - Scratch MLP_map3 1
wandb:  Model 2 - Pretrained DeBERTa_accuracy 0.6978
wandb:  Model 2 - Pretrained DeBERTa_macro_f1 0.695
wandb:      Model 2 - Pretrained DeBERTa_map3 0.80861
wandb: Model 3 - Similarity + LogReg_accu

In [10]:
# =====================================================================
# 8. MODEL 4 — WEIGHTED ENSEMBLE (validation-tuned weights)
# =====================================================================
print("\n--- Model 4: Weight-Searched Ensemble ---")

val_true = df_val['label'].values
best_map, best_weights = -1, (0.34, 0.33, 0.33)

step = 0.05
for w1 in np.arange(0, 1.0001, step):
    for w2 in np.arange(0, 1.0001 - w1, step):
        w3 = max(0.0, 1 - w1 - w2)   # clip tiny negative values from float rounding
        blend = w1 * scratch_val_preds + w2 * deberta_val_preds + w3 * logreg_val_preds
        score = compute_map_at_3(blend, val_true)
        if score > best_map:
            best_map, best_weights = score, (w1, w2, w3)

w_scratch, w_deberta, w_logreg = best_weights
best_blend_val = w_scratch * scratch_val_preds + w_deberta * deberta_val_preds + w_logreg * logreg_val_preds
m4_acc, m4_f1 = compute_cls_metrics(best_blend_val, val_true)

print(f"Best weights -> scratch: {w_scratch:.2f}, deberta: {w_deberta:.2f}, logreg: {w_logreg:.2f}")
print(f"Best validation MAP@3: {best_map:.5f}  (vs single-best model MAP@3 of "
      f"{max(m1_map, m2_map, m3_map):.5f})")

# Sanity guard: if the search leans almost entirely on one model, double check that
# model's train/val gap above before trusting this blend on the leaderboard --
# a lopsided weight is often a symptom of one model's validation score being
# unrealistically high (overfitting) rather than it being genuinely the best model.
if max(best_weights) > 0.7:
    print("NOTE: the weight search leaned heavily on a single model. If that model "
          "showed a large train/val gap above, treat this ensemble result with caution "
          "and consider re-running after checking for remaining leakage.")

results['Model 4 - Weighted Ensemble'] = {'accuracy': m4_acc, 'macro_f1': m4_f1, 'map3': best_map}
wandb.init(project="smart-mcq-solver-challenge", name="Model_4_Weighted_Ensemble", job_type="ensemble", reinit=True)
wandb.log({"ensemble_accuracy": m4_acc, "ensemble_macro_f1": m4_f1, "ensemble_map3": best_map,
           "w_scratch": w_scratch, "w_deberta": w_deberta, "w_logreg": w_logreg})
wandb.finish()

ensemble_test_probs = (w_scratch * scratch_test_preds) + (w_deberta * deberta_test_preds) + (w_logreg * logreg_test_preds)
final_ranked_preds = []
for row in ensemble_test_probs:
    top3_indices = np.argsort(-row)[:3]
    final_ranked_preds.append(' '.join([index_to_option[idx] for idx in top3_indices]))

submission = pd.DataFrame({'id': test_df['id'], 'Prediction': final_ranked_preds})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Pipeline complete. submission.csv successfully built.")
submission.head()

wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.



--- Model 4: Weight-Searched Ensemble ---
Best weights -> scratch: 0.20, deberta: 0.25, logreg: 0.55
Best validation MAP@3: 1.00000  (vs single-best model MAP@3 of 1.00000)


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260802_012038-fl7xfsln
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model_4_Weighted_Ensemble
wandb: ⭐️ View project at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge
wandb: 🚀 View run at https://wandb.ai/23f3000333-dl-genai-project/smart-mcq-solver-challenge/runs/fl7xfsln
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: ensemble_accuracy ▁
wandb: ensemble_macro_f1 ▁
wandb:     ensemble_map3 ▁
wandb:         w_deberta ▁
wandb:          w_logreg ▁
wandb:         w_scratch ▁
wandb: 
wandb: Run summary:
wandb: ensemble_accuracy 1
wandb: ensemble_macro_f1 1
wandb:     ensemble_map3 1
wandb:         w_deberta 0.25
wandb:          w_logreg 0.55
wandb:         w_scratch 0.2
wandb: 
wandb: 🚀 View run Model_4_Weighted_Ensemble at: https://wandb.ai/23f3000333-dl-ge

Pipeline complete. submission.csv successfully built.


,id,Prediction
0,1,A B D
1,2,B C D
2,3,B E C
3,4,E D C
4,5,C D B


In [11]:
# =====================================================================
# 9. SANITY CHECKS
# =====================================================================
assert len(submission) == len(test_df), "Row count mismatch with test set!"
assert submission['Prediction'].apply(lambda s: len(s.split()) == 3).all(), "Some rows don't have exactly 3 predictions!"
assert submission['Prediction'].apply(lambda s: all(c in option_to_index for c in s.split())).all(), "Invalid label found!"
print("All sanity checks passed. Ready to submit.")


All sanity checks passed. Ready to submit.
